# NovelReader v4.7.1 — Piper + VieNeu
Chọn engine ở cell 3. VieNeu tải 25 giọng Vbook và backbone/codec chính thức riêng, không chạy GGUF.


## 1. Khởi tạo môi trường & Mã nguồn

In [ ]:
import os, sys, subprocess, zipfile, stat, urllib.request, hashlib, json, shutil
from pathlib import Path

IS_KAGGLE = bool(os.getenv('KAGGLE_KERNEL_RUN_TYPE') or os.getenv('KAGGLE_URL_BASE') or Path('/kaggle/working').is_dir())
IS_COLAB = Path('/content').is_dir() and not IS_KAGGLE
BASE_DIR = Path('/kaggle/working') if IS_KAGGLE else (Path('/content') if IS_COLAB else Path.cwd())
BASE_DIR.mkdir(parents=True, exist_ok=True)
REPO_DIR = BASE_DIR / 'NovelReader_T4_Web'
# Reuse legacy data instead of silently creating an empty library on Kaggle.
legacy = Path('/content/NovelReader_T4_Web')
if IS_KAGGLE and (legacy / 'data/novel.sqlite3').is_file():
    REPO_DIR = legacy

REPO_DIR.mkdir(parents=True, exist_ok=True)

# BUNDLED_SOURCE_OVERLAY_V471
# v4.7.1 source only; never overwrite runtime data, model weights, or secrets.
import base64 as _overlay_b64, io as _overlay_io
_overlay_payload = '...'
if __import__("hashlib").sha256(_overlay_b64.b64decode(_overlay_payload)).hexdigest() != '2bb31347a46cd06bd399f9243ecb176da6bc89472040e8d4c88082e309ab492a':
    raise RuntimeError("Nguồn tích hợp v4.7.1 sai checksum.")
with zipfile.ZipFile(_overlay_io.BytesIO(_overlay_b64.b64decode(_overlay_payload))) as _overlay_zip:
    for _overlay_member in _overlay_zip.infolist():
        _overlay_parts = Path(_overlay_member.filename).parts
        if not _overlay_parts or '..' in _overlay_parts or Path(_overlay_member.filename).is_absolute() or '\\' in _overlay_member.filename:
            continue
        if _overlay_parts[0] in {'data','models','output_text','output_audio'} or any(p.startswith('.') for p in _overlay_parts):
            continue
        if _overlay_member.is_dir():
            continue
        _overlay_target = REPO_DIR / Path(*_overlay_parts)
        _overlay_target.parent.mkdir(parents=True, exist_ok=True)
        with _overlay_zip.open(_overlay_member) as _overlay_src, _overlay_target.open('wb') as _overlay_dst:
            shutil.copyfileobj(_overlay_src, _overlay_dst)
# BEGIN_RETIRED_STORAGE_CLEANUP
for _retired in ('gdrive/__init__.py', 'gdrive/api_drive.py', 'gdrive/colab_bridge.py', 'gdrive/export_service.py', 'gdrive/gdrive_service.py', 'storage/pixeldrain_service.py', 'web/routes/gdrive.py', 'web/routes/drive_api.py', 'web/routes/drive_exports.py', 'web/routes/pixeldrain_routes.py'):
    (REPO_DIR / _retired).unlink(missing_ok=True)
# END_RETIRED_STORAGE_CLEANUP
del _overlay_payload
# END_BUNDLED_SOURCE_OVERLAY_V471
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
for sub in ('data/logs', 'models/translation', 'models/voices', 'output_text', 'output_audio'):
    (REPO_DIR / sub).mkdir(parents=True, exist_ok=True)
print('🚀 Sẵn sàng tại:', REPO_DIR)
subprocess.run(['nvidia-smi'], check=False)


## 2. Cài đặt thư viện & Kích hoạt Piper CUDA (ORT-GPU + cuDNN)

In [ ]:
import subprocess, sys, os

print('📦 1/3: Cài đặt các gói phụ thuộc từ requirements.txt...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=False)

print('📦 2/3: Cài đặt ONNX Runtime GPU và bộ thư viện NVIDIA cuDNN 12...')
# Gỡ onnxruntime CPU và đảm bảo bản GPU được cài đặt chuẩn
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'], check=False, stdout=subprocess.DEVNULL)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'onnxruntime-gpu',
    'nvidia-cudnn-cu12',
    'nvidia-cublas-cu12',
    'nvidia-cuda-runtime-cu12',
    'ctranslate2>=4.5,<5'
], check=True)

print('📦 3/3: Kiểm tra các tiện ích audio hệ thống (ffmpeg, espeak-ng)...')
try:
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'espeak-ng'], check=False, stderr=subprocess.DEVNULL)
except Exception:
    pass

print('✅ Hoàn tất cài đặt thư viện!')

## 3. Cấu hình môi trường GPU & Bộ nhớ đệm

In [ ]:
# Chọn engine trước khi chạy cell này.
TTS_ENGINE = 'vieneu'  # 'piper' hoặc 'vieneu'
TTS_VOICE = 'truc_ly' if TTS_ENGINE == 'vieneu' else 'ngochuyennew'
VIENEU_DEVICE = 'cuda'  # 'cpu' nếu không có GPU
import os, site
IS_COLAB = globals().get("IS_COLAB", not globals().get("IS_KAGGLE", False))
from pathlib import Path

os.environ.update({
    'NOVEL_TTS_ENGINE':TTS_ENGINE,'NOVEL_TTS_VOICE':TTS_VOICE,
    'VIENEU_DEVICE':VIENEU_DEVICE,'VIENEU_MODEL_DIR':str(REPO_DIR/'models/vieneu'),
    'HACHIMIMT_WEB_BATCH_SIZE': '96',
    'HACHIMIMT_BATCH_SIZE': '96',
    'HACHIMIMT_CT2_BATCH_TYPE': 'tokens',
    'HACHIMIMT_CT2_WINDOW_MULTIPLIER': '16',
    'HACHIMIMT_INTER_THREADS': '1',
    'HACHIMIMT_COMPUTE_TYPE': 'int8_float16',
    'HACHIMIMT_FORCE_CT2_CUDA': '1',
    'HACHIMIMT_GPU_INDICES': '0',
    'HACHIMIMT_AUTO_ALL_GPUS': '0',
    'PIPER_DEVICE': 'cuda',
    'PIPER_REQUIRE_CUDA': '1',
    'PIPER_CUDNN_ALGO': 'DEFAULT',
    'AUDIO_WORKERS': '2',
    'TTS_CHUNK_MAX_CHARS': '4000',
    'ALLOWED_HOSTS': '*',
    'ALLOW_TUNNEL': '1',
    'COLAB_GPU': '1',
    'PYTHONUNBUFFERED': '1',
    'NOVEL_DATA': str(REPO_DIR / 'data'),
    'NOVEL_MODELS': str(REPO_DIR / 'models/translation'),
    'TTS_VOICE_DIR': str(REPO_DIR / 'models/voices'),
    'NOVEL_OUTPUT_TEXT': str(REPO_DIR / 'output_text'),
    'NOVEL_OUTPUT_AUDIO': str(REPO_DIR / 'output_audio'),
})

# Nạp thư viện cuDNN động vào LD_LIBRARY_PATH
nvidia_libs = []
for base in site.getsitepackages():
    nv = Path(base) / 'nvidia'
    if nv.is_dir():
        nvidia_libs.extend(str(p) for p in nv.glob('*/lib') if p.is_dir())

os.environ['LD_LIBRARY_PATH'] = ':'.join(nvidia_libs + [os.environ.get('LD_LIBRARY_PATH', '')])
print('✅ Đã nạp cấu hình tối ưu GPU thành công!')
if VIENEU_DEVICE=='cpu':
    os.environ.update({'HACHIMIMT_FORCE_CT2_CUDA':'0','HACHIMIMT_COMPUTE_TYPE':'default','PIPER_DEVICE':'cpu','PIPER_REQUIRE_CUDA':'0','COLAB_GPU':'0','AUDIO_WORKERS':'1'})


## 4. Tự tải JSON, model theo bộ đọc và kiểm tra GPU


In [ ]:
cmd=[sys.executable,str(REPO_DIR/'scripts/setup_tts.py'),'--engine',TTS_ENGINE,'--voice',TTS_VOICE]
if TTS_ENGINE=='vieneu': cmd+=['--device',VIENEU_DEVICE,'--install','--preload']
subprocess.run(cmd,cwd=str(REPO_DIR),env=dict(os.environ),check=True)
if VIENEU_DEVICE=='cuda':
    subprocess.run([sys.executable,'scripts/check_gpu.py','--model','HachimiMT-60-QT','--voice',TTS_VOICE,'--tts-engine',TTS_ENGINE],check=True)


## 5. Khởi chạy Web Server & Kết nối Tunnel Đa Nền Tảng
Tự động chọn tunnel khả dụng (Cloudflare hoặc Pinggy port 443) đảm bảo link web luôn hoạt động.

In [ ]:
import time, re, socket, urllib.request, secrets, subprocess
from IPython.display import display, Markdown

# Dọn dẹp tiến trình cũ nếu có
subprocess.run(['pkill', '-f', 'uvicorn.*main:app'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
time.sleep(2)

token = secrets.token_urlsafe(32)
os.environ['NOVEL_WEB_ACCESS_TOKEN'] = token
os.environ['NOVEL_EXPORT_SESSION_ID'] = secrets.token_hex(16)

server_log = REPO_DIR / 'data/logs/colab-server.log'
tunnel_log = REPO_DIR / 'data/logs/colab-tunnel.log'

# 1. Khởi động Uvicorn
print('🚀 Khởi động máy chủ NovelReader...')
with server_log.open('w') as sf:
    server_proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'main:app', '--host', '127.0.0.1', '--port', '8000',
         '--workers', '1', '--no-access-log', '--timeout-graceful-shutdown', '120'],
        cwd=str(REPO_DIR), stdout=sf, stderr=subprocess.STDOUT, env=dict(os.environ)
    )

for _ in range(120):
    if server_proc.poll() is not None:
        raise RuntimeError(f'Server không khởi động được:\n{server_log.read_text()[-2000:]}')
    try:
        req = urllib.request.Request('http://127.0.0.1:8000/api/state', headers={'X-Novel-Access': token})
        with urllib.request.urlopen(req, timeout=2):
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Hết thời gian chờ Uvicorn sẵn sàng.')
print('✅ Server Uvicorn đã sẵn sàng trên 127.0.0.1:8000!')

# 2. Kiểm tra xem port 7844 (Cloudflare) có bị chặn không
cf_blocked = True
try:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(3)
        if s.connect_ex(('198.41.200.233', 7844)) == 0:
            cf_blocked = False
except Exception:
    pass

public_url = None
tunnel_name = ''

if not cf_blocked:
    # Dùng Cloudflare nếu không bị chặn (Colab / Kaggle)
    print('🌐 Đang kết nối qua Cloudflare Tunnel...')
    cf_bin = BASE_DIR / 'cloudflared'
    if not cf_bin.is_file():
        urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cf_bin)
        cf_bin.chmod(0o755)
    with tunnel_log.open('w') as tf:
        tunnel_proc = subprocess.Popen([str(cf_bin), 'tunnel', '--protocol', 'http2', '--url', 'http://127.0.0.1:8000'], stdout=tf, stderr=subprocess.STDOUT)
    for _ in range(60):
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', tunnel_log.read_text())
        if match:
            public_url = match.group(0)
            tunnel_name = 'Cloudflare'
            break
        time.sleep(1)

if not public_url:
    # Fallback Pinggy qua port 443 (đặc biệt cho Molab nơi port 7844 bị chặn)
    print('🌐 Port 7844 bị chặn — tự động kết nối qua Pinggy HTTPS Tunnel (Port 443)...')
    tunnel_proc = subprocess.Popen(
        ['ssh', '-p', '443', '-o', 'StrictHostKeyChecking=no', '-R', '0:localhost:8000', 'a.pinggy.io'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ, 'TERM': 'dumb'}
    )
    start_t = time.time()
    while time.time() - start_t < 30:
        line = tunnel_proc.stdout.readline()
        if not line: break
        match = re.search(r'(https://[a-zA-Z0-9.-]+\.pinggy[a-zA-Z0-9.-]*\.link)', line)
        if match:
            public_url = match.group(1)
            tunnel_name = 'Pinggy (Port 443)'
            break

if public_url:
    full_link = f'{public_url}/?access_token={token}'
    display(Markdown(f'## 🎉 [BẤM VÀO ĐÂY ĐỂ MỞ NOVELREADER]({full_link})\n*(Tunnel: {tunnel_name})*'))
    print(f'\n🔗 Link truy cập:\n{full_link}\n')
    print(f'🔑 Access Token: {token}\n')
    print('⚠️ Giữ tab này mở trong suốt quá trình sử dụng.')
else:
    print('⚠️ Không tạo được tunnel công khai. Bạn có thể truy cập nội bộ qua port 8000.')

# Vòng lặp hiển thị log
try:
    with server_log.open() as f:
        while server_proc.poll() is None:
            line = f.readline()
            if line: print(line, end='')
            else: time.sleep(0.5)
except KeyboardInterrupt:
    print('Đang dừng server...')
    server_proc.terminate()
    if 'tunnel_proc' in locals(): tunnel_proc.terminate()

## 6. Sao lưu kết quả dịch & Audio
Chạy cell này sau khi dừng Cell 5 để nén toàn bộ database và file đã dịch.

In [ ]:
from datetime import datetime
import sqlite3, tempfile

backup_name = BASE_DIR / f'NovelReader_Backup_{datetime.now().strftime("%Y%m%d_%H%M%S")}.zip'
with zipfile.ZipFile(backup_name, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ('output_text', 'output_audio'):
        p_folder = REPO_DIR / folder
        if p_folder.is_dir():
            for p in p_folder.rglob('*'):
                if p.is_file() and not any(part.startswith('.') for part in p.relative_to(REPO_DIR).parts):
                    z.write(p, p.relative_to(REPO_DIR))
    db = REPO_DIR / 'data/novel.sqlite3'
    if db.exists():
        with tempfile.TemporaryDirectory() as temp:
            snapshot = Path(temp) / 'novel.sqlite3'
            with sqlite3.connect(db) as src, sqlite3.connect(snapshot) as dst:
                src.backup(dst)
            z.write(snapshot, 'data/novel.sqlite3')

print(f'✅ Đã tạo file sao lưu: {backup_name.name} ({backup_name.stat().st_size / (1024*1024):.1f} MB)')
if IS_COLAB:
    from google.colab import files
    files.download(str(backup_name))
elif IS_KAGGLE:
    from IPython.display import FileLink
    display(FileLink(str(backup_name)))